# 14 — Unit Testing, Fixtures, and Mocking

Tests turn assumptions into executable checks. Python's built-in unittest framework provides assertions, setup hooks, and mocks. External pytest can be introduced later without making the notebooks depend on it.

*Intermediate · 75–120 min*

## You will learn

- Structure unittest.TestCase assertions
- Test success, invalid input, and boundaries
- Use setUp and temporary resources
- Mock external boundaries without calling the network

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. Writing a first unit test

A test method begins with test_ and calls assertions such as assertEqual and assertRaises. In Jupyter, use a TestLoader and TextTestRunner rather than unittest.main, because the notebook kernel supplies its own command-line arguments.

In [ ]:
import unittest

def normalize_name(text):
    if not text.strip():
        raise ValueError("empty name")
    return text.strip().title()

class NameTests(unittest.TestCase):
    def test_normalization(self):
        self.assertEqual(normalize_name("  ada "), "Ada")
    def test_reject_empty(self):
        with self.assertRaises(ValueError):
            normalize_name("  ")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(NameTests)
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful()

**Think about it:** What does assertRaises test that assertEqual cannot?

### 2. Setup and state isolation

Each test should operate on fresh state to avoid execution-order dependence. setUp runs before each test method; tearDown can clean resources. Prefer tiny fixtures and temporary directories for files.

In [ ]:
import unittest

class Basket:
    def __init__(self):
        self.items = []
    def add(self, price):
        self.items.append(price)
    def total(self):
        return sum(self.items)

class BasketTests(unittest.TestCase):
    def setUp(self):
        self.basket = Basket()
    def test_empty(self):
        self.assertEqual(self.basket.total(), 0)
    def test_two_items(self):
        self.basket.add(3)
        self.basket.add(4)
        self.assertEqual(self.basket.total(), 7)

assert unittest.TextTestRunner(verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(BasketTests)
).wasSuccessful()

**Think about it:** Why would sharing one Basket across all tests introduce flaky results?

### 3. Mocking a dependency

A mock substitutes a collaborator that would otherwise perform I/O or nondeterministic work. Test observable behavior and important interactions without over-specifying implementation details. MagicMock can count calls and set return values.

In [ ]:
from unittest.mock import Mock

def send_report(sender, text):
    if not text:
        raise ValueError("report is empty")
    return sender.send(text)

fake_sender = Mock()
fake_sender.send.return_value = "queued"
assert send_report(fake_sender, "Results ready") == "queued"
fake_sender.send.assert_called_once_with("Results ready")
print(fake_sender.send.call_count)

**Think about it:** Which part of this code would be replaced by a real email or HTTP service?

### 4. Subtests and test matrices

subTest lets a single test method check many inputs while reporting which case failed. Choose meaningful boundary data rather than only ordinary examples. Avoid tests that depend on the current date or network.

In [ ]:
import unittest

def clamp(value, low, high):
    if low > high:
        raise ValueError("invalid range")
    return max(low, min(value, high))

class ClampTests(unittest.TestCase):
    def test_cases(self):
        for value, expected in [(-10, 0), (5, 5), (15, 10)]:
            with self.subTest(value=value):
                self.assertEqual(clamp(value, 0, 10), expected)

assert unittest.TextTestRunner(verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(ClampTests)
).wasSuccessful()

**Think about it:** Why is this more informative than testing only the value 5?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/03_data_quality_and_testing_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/14', {1: 'd8ca98a22b21855f', 2: '9697d126851d122e', 3: '257fd55ff4ccbda1', 4: 'dfff4e41bd506f38'})

**Exercise 1.** Complete the test for a discount validator, then run it.

In [ ]:
import io, unittest

def validate_discount(percent):
    if not 0 <= percent <= 100:
        raise ValueError("bad percent")
    return percent

class DiscountTest(unittest.TestCase):
    def test_valid(self):
        self.assertEqual(validate_discount(10), ___)

    def test_invalid(self):
        with self.___(ValueError):
            validate_discount(150)

suite = unittest.TestLoader().loadTestsFromTestCase(DiscountTest)
run_result = unittest.TextTestRunner(verbosity=0, stream=io.StringIO()).run(suite)
check(1, (run_result.testsRun, len(run_result.failures) + len(run_result.errors)))

**Exercise 2.** `setUp` gives every test a fresh cart.

In [ ]:
import io, unittest

class CartTest(unittest.TestCase):
    def setUp(self):
        self.cart = ___

    def test_add(self):
        self.cart.append("pen")
        self.assertEqual(len(self.cart), 1)

    def test_starts_empty(self):
        self.assertEqual(self.cart, ___)

suite = unittest.TestLoader().loadTestsFromTestCase(CartTest)
r = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(suite)
check(2, (r.testsRun, r.wasSuccessful()))

**Exercise 3.** Mock a notification sender and check what it was called with.

In [ ]:
from unittest.mock import Mock

def notify_overdue(sender, user):
    sender.send(user, f"{user}, your book is overdue")

sender = ___
notify_overdue(sender, "ana")
check(3, sender.send.call_args.args)

**Exercise 4.** Turn a table of edge cases into `subTest` checks.

In [ ]:
import io, unittest

def is_even(n):
    return n % 2 == 0

class EvenTest(unittest.TestCase):
    def test_table(self):
        for value, expected in [(0, True), (1, False), (-2, ___), (7, ___)]:
            with self.___(value=value):
                self.assertEqual(is_even(value), expected)

r = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(unittest.TestLoader().loadTestsFromTestCase(EvenTest))
check(4, r.wasSuccessful())

## Common mistakes

- Use unittest's loader and runner inside notebooks; unittest.main reads notebook arguments.
- Do not depend on real network services in unit tests.
- Tests should be deterministic and isolated from each other.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Where does a unittest.TestCase prepare fresh data for each test?

   - **a)** __init__
   - **b)** setUp
   - **c)** tearDownClass

**2.** Why mock an HTTP client in unit tests?

   - **a)** To hide bugs
   - **b)** Tests become fast, offline and deterministic
   - **c)** Mocks are more accurate

**3.** Which assertion checks that an error is raised?

   - **a)** assertEqual
   - **b)** assertRaises
   - **c)** assertTrue

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/14', {1: '412e05a94435', 2: '277d34fca480', 3: 'fcaedd512825'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [15 — SQLite, SQL, and Transactions](15_sqlite_and_transactions.ipynb)